In [ ]:
from flask import Flask, jsonify, request
import torch
import joblib
from transformers import AutoTokenizer
import javalang
import esprima
import re

# loading trained models and tokenizer

data_class = joblib.load("./java_data_class")
feature_envy = joblib.load("./java_feature_envy")
java_long_parameter = joblib.load("./java_long_parameter")
java_long_method = joblib.load("./java_long_method")
javascript_graphcodebert = joblib.load("./javascript_graphcodebert")
javascript_complex_conditional = joblib.load("./javascript_complex_conditional")
java_blob = joblib.load("./java_blob")
js_callback_graphcodebert = joblib.load("./js_callback_graphcodebert")
js_strictequality_codebert = joblib.load("./js_strictequality_codebert")
js_impropererror_codebert = joblib.load("./js_impropererror_codebert")
tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
gcb_tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
codebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

data_class.eval()
feature_envy.eval()
java_long_method.eval()
java_long_parameter.eval()
javascript_graphcodebert.eval()
javascript_complex_conditional.eval()
java_blob.eval()
js_callback_graphcodebert.eval()
js_strictequality_codebert.eval()
js_impropererror_codebert.eval()


# defining some of the helper functions

def extract_java_functions(java_code):
    try:
        parsed_code = javalang.parse.parse(java_code)
    except javalang.parser.JavaSyntaxError as e:
        return -1

    extracted_functions = []
    lines = java_code.split('\n')

    for _, node in parsed_code.filter(javalang.tree.MethodDeclaration):
        start_position = node.position
        if start_position is None:
            continue
            
        start_line = start_position.line - 1
        start_col = start_position.column - 1

        method_lines = []
        brace_count = 0
        inside_method = False

        for i in range(start_line, len(lines)):
            line = lines[i]
            if not inside_method:
                method_lines.append(line)
                if line.strip().endswith("{"):
                    inside_method = True
                    brace_count = 1
            else:
                method_lines.append(line)
                brace_count += line.count('{')
                brace_count -= line.count('}')
                if brace_count == 0:
                    break

        function_code = "\n".join(method_lines)
        extracted_functions.append(function_code.strip())
        
    return extracted_functions


def removeCommentsForJava(java_code):
    code_without_multiline_comments = re.sub(r'/\*.*?\*/', '', java_code, flags=re.DOTALL)
    code_without_comments = re.sub(r'//.*', '', code_without_multiline_comments)
    cleaned_code = '\n'.join([line for line in code_without_comments.split('\n') if line.strip() != ''])
    return cleaned_code

def get_java_function_name(java_code):
    pattern = re.compile(r'\b(?:public|private|protected|static|final|abstract|synchronized)?\s*[\w<>,]+\s+(\w+)\s*\(([^)]*)\)', re.MULTILINE)
    matches = pattern.findall(java_code)
    for match in matches:
        function_name, parameters = match
        return function_name + "(" + parameters + ")"
    

def extract_javascript_functions(code):
    try:
        ast = esprima.parseScript(code, options={'tolerant': True, 'range': True})
        functions = []

        def traverse(node):
            # Check if the node is a FunctionDeclaration with a name
            if node.type == 'FunctionDeclaration' and node.id:
                start, end = node.range
                functions.append(code[start:end])

            # Check if the node is a VariableDeclarator with a function assigned to it
            elif node.type == 'VariableDeclarator' and node.init and node.init.type in ['FunctionExpression', 'ArrowFunctionExpression']:
                # This is a function assigned to a variable, so we consider it "named"
                start, end = node.range
                functions.append(code[start:end])

            # Recurse through child nodes
            for key, value in vars(node).items():
                if isinstance(value, esprima.nodes.Node):
                    traverse(value)
                elif isinstance(value, list):
                    for item in value:
                        if isinstance(item, esprima.nodes.Node):
                            traverse(item)

        traverse(ast)
        return functions

    except Exception as e:
        return -1
    

def contains_jsx(code):
    jsx_pattern = re.compile(r'<[a-zA-Z][^>]*?>')
    return bool(jsx_pattern.search(code))

def remove_jsx_return_blocks(code):
    lines = code.splitlines()
    new_code = []
    in_return_block = False
    paren_count = 0
    
    for line in lines:
        stripped_line = line.strip()
        if 'return' in stripped_line and '(' in stripped_line and not in_return_block:
            in_return_block = True
            paren_count = stripped_line.count('(') - stripped_line.count(')')
        elif in_return_block:
            paren_count += stripped_line.count('(') - stripped_line.count(')')
            
            if paren_count == 0:
                in_return_block = False
            continue
        else:
            new_code.append(line)
    
    return '\n'.join(new_code).strip()


def remove_comments_and_blank_lines_js(js_code):
    no_single_line_comments = re.sub(r'//.*', '', js_code)
    no_comments = re.sub(r'/\*.*?\*/', '', no_single_line_comments, flags=re.DOTALL)
    cleaned_code = "\n".join([line.strip() for line in no_comments.split('\n') if line.strip() != ''])
    return cleaned_code

def check(js_code):
    loose_equality_pattern = re.compile(r'(?<![=!])==([^=]|$)')
    loose_inequality_pattern = re.compile(r'!=([^=]|$)')
    lines = js_code.splitlines()
    for line in lines:
        if loose_equality_pattern.search(line) or loose_inequality_pattern.search(line):
            return True
    return False

def split_sequence_into_chunks(sequence, max_length=512):
    gcb_tokenizer.model_max_length = 1e12
    special_tokens_count = gcb_tokenizer.num_special_tokens_to_add(pair=False)
    # Adjust the maximum length to account for special tokens
    adjusted_max_length = max_length - special_tokens_count
    # Tokenize the entire sequence without adding special tokens
    tokens = gcb_tokenizer.encode(sequence, add_special_tokens=False)

    chunks = []
    for i in range(0, len(tokens), adjusted_max_length):
        chunk_tokens = tokens[i:i + adjusted_max_length]
        chunk_text = gcb_tokenizer.decode(chunk_tokens, clean_up_tokenization_spaces=False)
        chunks.append(chunk_text)
    return chunks


# creating the main server instance
server = Flask(__name__)


# routes of server

#route for java files
@server.route('/java', methods=['POST'])
def scanJava():
    try:
        file = request.get_json()
        filename = next(iter(file))
        response = {}
        innerjson = {}
        functions = extract_java_functions(file[filename])
        if functions == -1:
           raise Exception("Error")
        else:
            for function in functions:
                function_name = get_java_function_name(function)
                code_smells = []
                tokenizedInput = tokenizer(function, padding=True,truncation=True, return_tensors="pt")
                input_ids = tokenizedInput['input_ids'].to("cpu")
                attention_mask = tokenizedInput['attention_mask'].to("cpu")

                with torch.no_grad():
                    long_parameter_outputs = java_long_parameter(input_ids=input_ids, attention_mask=attention_mask)
                    long_method_outputs = java_long_method(input_ids=input_ids, attention_mask=attention_mask)
                    feature_envy_outputs = feature_envy(input_ids=input_ids, attention_mask=attention_mask)

                long_parameter_logits = long_parameter_outputs.logits
                long_method_logits = long_method_outputs.logits
                feature_envy_logits = feature_envy_outputs.logits
            
                long_parameter_probabilities = torch.softmax(long_parameter_logits, dim=-1)
                long_method_probabilities = torch.softmax(long_method_logits, dim=-1)
                feature_envy_probabilities = torch.softmax(feature_envy_logits, dim=-1)
            
                long_parameter_predicted_labels = torch.argmax(long_parameter_probabilities, dim=-1)
                long_method_predicted_labels = torch.argmax(long_method_probabilities, dim=-1)
                feature_envy_predicted_labels = torch.argmax(feature_envy_probabilities, dim=-1)
            
                if long_parameter_predicted_labels.item() == 1:
                    code_smells.append("Long Parameters")
                    code_smells.append("Primitive Obsession")
                
                if long_method_predicted_labels.item() == 1:
                    code_smells.append("Long Method")
                
                if feature_envy_predicted_labels.item() == 1:
                    code_smells.append("Feature Envy")
                
                if len(code_smells) > 0:
                    innerjson[function_name] = code_smells
                    
            # Now feeding the whole file to identify file related code smells
            cleanFileCode = removeCommentsForJava(file[filename])
            tokenizedInput = tokenizer(cleanFileCode, padding=True,truncation=True, return_tensors="pt")
            input_ids = tokenizedInput['input_ids'].to("cpu")
            attention_mask = tokenizedInput['attention_mask'].to("cpu")
            with torch.no_grad():
                outputs = data_class(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)
            predicted_labels = torch.argmax(probabilities, dim=-1)
            
            if predicted_labels.item() == 1:
                innerjson["File"] = "Data Class"
            else:
                # Now checking for blob/large class
                tokenizedInput = tokenizer(file[filename], padding=True,truncation=True, return_tensors="pt")
                input_ids = tokenizedInput['input_ids'].to("cpu")
                attention_mask = tokenizedInput['attention_mask'].to("cpu")
                with torch.no_grad():
                    outputs = java_blob(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                probabilities = torch.softmax(logits, dim=-1)
                predicted_labels = torch.argmax(probabilities, dim=-1)
                if predicted_labels.item() == 1:
                    innerjson["File"] = "blob/Large Class"
                
                        
            if not innerjson:
                response[filename] = "clean"
                return jsonify(response),201
        
            else:
                response[filename] = innerjson
                return jsonify(response),201
            
    except Exception as e:
        response[filename] = "error"
        return jsonify(response),422
            

#route for javascript files
@server.route('/javascript', methods=['POST'])
def scanJavascript():
    try:
        file = request.get_json()
        filename = next(iter(file))
        javascript_code = file[filename]
        response = {}
        innerjson = {}
        
        #checking if the code contains jsx
        has_jsx = contains_jsx(javascript_code)
        if has_jsx:
            javascript_code = remove_jsx_return_blocks(javascript_code)
        
        #Now extracting the functions
        functions = extract_javascript_functions(javascript_code)
        
        if functions == -1:
           raise Exception("Error")
        else:
            count = 0
            for function in functions:
                code_smells = []
                code_smells.append(function)
                tokenizedInput = tokenizer(function, padding=True,truncation=True, return_tensors="pt")
                input_ids = tokenizedInput['input_ids'].to("cpu")
                attention_mask = tokenizedInput['attention_mask'].to("cpu")
                
                codebert_tokens = codebert_tokenizer(function, padding=True,truncation=True, return_tensors="pt")
                codebert_input_ids = codebert_tokens['input_ids'].to("cpu")
                codebert_attention_mask = codebert_tokens['attention_mask'].to("cpu")
                
                segments = split_sequence_into_chunks(function)
                smells_for_longer_sequence = []
                for chunk in segments:
                    tokens_for_chunk = tokenizer(chunk, padding=True, truncation=True, max_length=512, return_tensors="pt")
                    input_ids_for_chunk = tokens_for_chunk['input_ids'].to("cpu")
                    attention_mask_for_chunk = tokens_for_chunk['attention_mask'].to("cpu")
                    with torch.no_grad():
                        callbackhell_outputs = js_callback_graphcodebert(input_ids=input_ids_for_chunk, attention_mask=attention_mask_for_chunk)
                    callbackhell_logits = callbackhell_outputs.logits                
                    callbackhell_probabilities = torch.softmax(callbackhell_logits, dim=-1)
                    callbackhell_prediction = torch.argmax(callbackhell_probabilities, dim=-1)
                    smells_for_longer_sequence.append(callbackhell_prediction.item())
                        
                    

                with torch.no_grad():
                    outputs = javascript_graphcodebert(input_ids=input_ids, attention_mask=attention_mask)
                    complex_conditional_outputs = javascript_complex_conditional(input_ids=input_ids, attention_mask=attention_mask)
#                     callbackhell_outputs = js_callback_graphcodebert(input_ids=input_ids, attention_mask=attention_mask)
                    impropererror_outputs = js_impropererror_codebert(input_ids=codebert_input_ids, attention_mask=codebert_attention_mask)
                    strictequality_outputs = js_strictequality_codebert(input_ids=codebert_input_ids, attention_mask=codebert_attention_mask)

                complex_conditional_logits = complex_conditional_outputs.logits                
                complex_conditional_probabilities = torch.softmax(complex_conditional_logits, dim=-1)
                complex_conditional_prediction = torch.argmax(complex_conditional_probabilities, dim=-1)
                
#                 callbackhell_logits = callbackhell_outputs.logits                
#                 callbackhell_probabilities = torch.softmax(callbackhell_logits, dim=-1)
#                 callbackhell_prediction = torch.argmax(callbackhell_probabilities, dim=-1)
                
                impropererror_logits = impropererror_outputs.logits                
                impropererror_probabilities = torch.softmax(impropererror_logits, dim=-1)
                impropererror_prediction = torch.argmax(impropererror_probabilities, dim=-1)
                
                strictequality_logits = strictequality_outputs.logits                
                strictequality_probabilities = torch.softmax(strictequality_logits, dim=-1)
                strictequality_prediction = torch.argmax(strictequality_probabilities, dim=-1)
                
                
                logits = outputs.logits
                probabilities = torch.softmax(logits, dim=-1)
                long_parameter_prob = probabilities[:, 0]
                long_method_prob = probabilities[:, 1]
                empty_catch_prob = probabilities[:, 2]
                complex_method_prob = probabilities[:, 3]
                
                
                if complex_conditional_prediction.item() == 1:
                    code_smells.append("Complex Conditional")
                    
#                 if callbackhell_prediction.item() == 1:
#                     code_smells.append("Callback Hell")

                if sum(smells_for_longer_sequence)/len(smells_for_longer_sequence) > 0.65:
                    code_smells.append("Callback Hell")
                    
                if impropererror_prediction.item() == 1:
                    code_smells.append("Improper Error Handling")
                    
                if strictequality_prediction.item() == 1 and check(function):
                    code_smells.append("Strict Operator Smell")
                
                if (long_parameter_prob > 0.95):
                    code_smells.append("Long parameter")

                if (long_method_prob > 0.7 or complex_method_prob > 0.7):
                      code_smells.append("Long function")

                if (empty_catch_prob > 0.95):
                      code_smells.append("Empty catch")
                        
                if len(code_smells) > 1:
                    innerjson[count] = code_smells
                    count = count + 1
                        
            if not innerjson:
                response[filename] = "clean"
                return jsonify(response),201
        
            else:
                response[filename] = innerjson
                return jsonify(response),201
            
    except Exception as e:
        response[filename] = "error"
        return jsonify(response),422
            

# Running the server
if __name__ == '__main__':
    server.run(debug=True, port=9000, use_reloader=False)

C:\ProgramData\anaconda3\Lib\site-packages\transformers\utils\generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:9000
Press CTRL+C to quit
127.0.0.1 - - [03/Dec/2024 23:09:27] "POST /java HTTP/1.1" 201 -
127.0.0.1 - - [03/Dec/2024 23:09:40] "POST /java HTTP/1.1" 201 -
127.0.0.1 - - [03/Dec/2024 23:09:53] "POST /java HTTP/1.1" 201 -
127.0.0.1 - - [03/Dec/2024 23:10:16] "POST /java HTTP/1.1" 201 -
